# Security Log Analysis Pipeline
### APT29 Evaluation Dataset · Sysmon Events
**Pipeline**: Parse → Anomaly Detection → BERTopic → RAG + LLM (DSPy + Ollama)

> **Runtime**: Set Colab to **GPU** (T4 or A100) before running.  
> **Dataset**: Automatically fetched from the Git repository and extracted to `/content/data_path`.

---


## Setup — Install Dependencies

In [ ]:
# Install all pipeline dependencies
!pip install -q \
    pandas pyarrow ijson \
    scikit-learn umap-learn \
    plotly kaleido \
    bertopic sentence-transformers hdbscan \
    chromadb \
    langchain langchain-community \
    dspy-ai \
    mitreattack-python \
    nltk pyyaml regex requests tqdm

import nltk
nltk.download('stopwords', quiet=True)
print("All dependencies installed")


### Fetch and Extract Dataset

In [ ]:
import os
import zipfile
import glob

# URL to the dataset in your GitHub repo (e.g., a .zip file containing the JSON)
# Replace this with the actual URL to your dataset zip file
DATA_REPO_URL = "https://raw.githubusercontent.com/OTRF/Security-Datasets/master/datasets/compound/apt29/day1/apt29_evals_day1_manual.zip"
ZIP_PATH = "/content/apt29_evals_day1_manual.zip"
EXTRACT_DIR = "/content/data_path"

if not os.path.exists(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print(f"Downloading dataset from {DATA_REPO_URL}...")
    !wget -q {DATA_REPO_URL} -O {ZIP_PATH}
    
if os.path.exists(ZIP_PATH):
    if not zipfile.is_zipfile(ZIP_PATH):
        raise ValueError(f"The downloaded file is not a valid zip file! Did you forget to update the placeholder DATA_REPO_URL?\nCurrent URL: {DATA_REPO_URL}")
    print("Extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)

# Dynamically find the extracted JSON file to use in the pipeline
json_files = glob.glob(f"{EXTRACT_DIR}/**/*.json", recursive=True)
if json_files:
    DATA_PATH = json_files[0]
    print(f"Found dataset: {DATA_PATH}")
else:
    # Fallback to the default expected path
    DATA_PATH = f"{EXTRACT_DIR}/apt29_evals_day1_manual_2020-05-01225525.json"
    print(f"No JSON found dynamically, falling back to: {DATA_PATH}")


### Global configuration

In [ ]:
# ─── EDIT THESE PATHS IF NEEDED ───────────────────────────────────────────
# DATA_PATH is set dynamically above, but we keep a fallback just in case
if 'DATA_PATH' not in locals():
    DATA_PATH = "/content/data_path/apt29_evals_day1_manual_2020-05-01225525.json"

NORMALIZED_PARQUET = "/content/data/normalized.parquet"
ANOMALIES_PARQUET  = "/content/data/anomalies.parquet"
TOPICS_PARQUET     = "/content/data/anomalies_with_topics.parquet"
CHROMA_DIR         = "/content/data/chroma_db"
RESULTS_JSON       = "/content/data/llm_results.json"
TOPIC_MODEL_DIR    = "/content/data/bertopic_model"

OLLAMA_MODEL       = "llama3"       # or "mistral", "phi3"
CONTAMINATION      = 0.05           # fraction flagged as anomalous
TOP_N_TOPICS       = 12             # topics passed to LLM
EVENTS_PER_TOPIC   = 3              # worst anomalies per topic

import os
os.makedirs("/content/data", exist_ok=True)
print("Config ready")


---
## Stage 1 — Parse & Normalize

Streams the 385 MB NDJSON file line-by-line (no OOM), normalises each Sysmon
event into a flat schema, engineers ML features, and saves `normalized.parquet`.

**Key features extracted**:
| Feature | Description |
|---|---|
| `event_id` | Sysmon event type (10=ProcessAccess, 11=FileCreate, 13=RegistrySet …) |
| `process_depth` | Depth of the process image path |
| `granted_access` | Hex access rights → int |
| `is_system` | Is the account NT AUTHORITY\SYSTEM? |
| `hour_of_day` / `day_of_week` | Temporal features |
| `message_len` | Raw message character count |
| `eid_*` | One-hot top-15 EventIDs |


In [ ]:
"""
Stage 1: Parse & Normalize
Streams the NDJSON log file in chunks and produces a normalized Parquet file.
"""
import json, os, re
import pandas as pd
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm


# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH         = "/content/data_path/apt29_evals_day1_manual_2020-05-01225525.json"
NORMALIZED_PARQUET = "/content/data/normalized.parquet"
CHUNK_SIZE        = 50_000          # rows kept in memory before flushing
# ──────────────────────────────────────────────────────────────────────────────


def _safe_int(val, default=0):
    try:
        return int(val)
    except (TypeError, ValueError):
        return default


def _hex_to_int(val):
    try:
        return int(val, 16) if isinstance(val, str) and val.lower().startswith("0x") else 0
    except ValueError:
        return 0


def _parse_dt(s):
    try:
        return datetime.strptime(s, "%Y-%m-%d %H:%M:%S")
    except Exception:
        return None


def _process_depth(path: str) -> int:
    return path.count("\\") if path else 0


def _basename(path: str) -> str:
    return path.split("\\")[-1].lower() if path else ""


def normalize_event(ev: dict) -> dict:
    """Flatten one raw Sysmon JSON event into a ML-ready dict."""
    dt   = _parse_dt(ev.get("EventTime", ""))
    img  = ev.get("Image") or ev.get("SourceImage") or ""
    timg = ev.get("TargetImage") or ""
    tobj = ev.get("TargetObject") or ""
    tfn  = ev.get("TargetFilename") or ""
    msg  = ev.get("Message") or ""

    return {
        # ── identifiers ──────────────────────────────────────────────────────
        "record_number"    : _safe_int(ev.get("RecordNumber")),
        "event_time"       : ev.get("EventTime", ""),
        "timestamp"        : dt.isoformat() if dt else "",
        "hour_of_day"      : dt.hour        if dt else -1,
        "day_of_week"      : dt.weekday()   if dt else -1,
        # ── event metadata ───────────────────────────────────────────────────
        "event_id"         : _safe_int(ev.get("EventID")),
        "channel"          : ev.get("Channel", ""),
        "source_name"      : ev.get("SourceName", ""),
        "severity"         : ev.get("Severity", ""),
        "severity_value"   : _safe_int(ev.get("SeverityValue")),
        "hostname"         : ev.get("Hostname", ""),
        # ── identity ─────────────────────────────────────────────────────────
        "account_name"     : ev.get("AccountName", ""),
        "domain"           : ev.get("Domain", ""),
        "user_id"          : ev.get("UserID", ""),
        "is_system"        : 1 if ev.get("AccountName", "").upper() == "SYSTEM" else 0,
        # ── process / image ──────────────────────────────────────────────────
        "image"            : img,
        "image_base"       : _basename(img),
        "process_depth"    : _process_depth(img),
        "process_id"       : _safe_int(ev.get("ProcessId") or ev.get("SourceProcessId")),
        "target_image"     : timg,
        "target_image_base": _basename(timg),
        # ── access / registry / file ─────────────────────────────────────────
        "granted_access"   : _hex_to_int(ev.get("GrantedAccess", "0x0")),
        "target_object"    : tobj,
        "target_filename"  : tfn,
        # ── text ─────────────────────────────────────────────────────────────
        "message"          : msg,
        "message_len"      : len(msg),
        "call_trace"       : ev.get("CallTrace", ""),
        "rule_name"        : ev.get("RuleName", ""),
    }


def parse_stage(data_path: str = DATA_PATH,
                out_path: str  = NORMALIZED_PARQUET,
                chunk_size: int = CHUNK_SIZE) -> pd.DataFrame:
    """
    Streams the NDJSON file line-by-line, normalizes each event,
    and saves a single consolidated Parquet file.
    Returns the final DataFrame.
    """
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    chunks, records, errors, total = [], [], 0, 0

    print(f"Streaming: {data_path}")
    with open(data_path, "r", encoding="utf-8", errors="replace") as fh:
        for line in tqdm(fh, desc="Parsing events", unit=" lines"):
            line = line.strip()
            if not line:
                continue
            try:
                ev = json.loads(line)
                records.append(normalize_event(ev))
                total += 1
            except json.JSONDecodeError:
                errors += 1
                continue

            if len(records) >= chunk_size:
                chunks.append(pd.DataFrame(records))
                records = []

    if records:
        chunks.append(pd.DataFrame(records))

    print(f"\nParsed {total:,} events | JSON errors: {errors:,}")

    df = pd.concat(chunks, ignore_index=True)

    # ── post-parse feature engineering ───────────────────────────────────────
    # One-hot top-15 EventIDs
    top_eids = df["event_id"].value_counts().head(15).index.tolist()
    for eid in top_eids:
        df[f"eid_{eid}"] = (df["event_id"] == eid).astype(np.int8)

    # Channel bucket
    df["channel_bucket"] = df["channel"].str.extract(r"(Sysmon|Security|System|Application)",
                                                       expand=False).fillna("Other")

    df.to_parquet(out_path, index=False)
    print(f"Saved -> {out_path}  ({df.shape[0]:,} rows x {df.shape[1]} cols)")
    return df

In [ ]:
df_norm = parse_stage(
    data_path  = DATA_PATH,
    out_path   = NORMALIZED_PARQUET,
    chunk_size = 50_000,
)
df_norm.head(3)


In [ ]:
import pandas as pd
import plotly.express as px

df_norm = pd.read_parquet(NORMALIZED_PARQUET)

fig = px.histogram(
    df_norm, x="event_id",
    color="is_system",
    title="Event ID Distribution (System vs User Accounts)",
    template="plotly_dark",
    barmode="overlay",
    opacity=0.8,
    height=420,
)
fig.show()

print(f"Total events : {len(df_norm):,}")
print(f"Unique hosts : {df_norm['hostname'].nunique()}")
print(f"Event ID types: {df_norm['event_id'].nunique()}")
print(df_norm[['event_id','hostname','channel','severity','message_len']].describe())


---
## Stage 2 — Anomaly Detection (Isolation Forest + UMAP)

Trains an **Isolation Forest** on the engineered feature matrix (no labels needed).
Events in the bottom `CONTAMINATION` percentile are flagged as anomalous.
A UMAP 2-D projection is rendered as an interactive scatter plot.


In [ ]:
"""
Stage 2: Anomaly Detection
Loads normalized.parquet, scores every event with Isolation Forest,
projects to 2-D via UMAP, and saves anomalies.parquet.
"""
import os
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import umap

# ── Config ────────────────────────────────────────────────────────────────────
NORMALIZED_PARQUET  = "/content/data/normalized.parquet"
ANOMALIES_PARQUET   = "/content/data/anomalies.parquet"
CONTAMINATION       = 0.05          # expected anomaly fraction
UMAP_SAMPLE         = 60_000        # rows sent to UMAP (memory guard)
RANDOM_STATE        = 42
# ──────────────────────────────────────────────────────────────────────────────


def build_feature_matrix(df: pd.DataFrame):
    """
    Returns (X_scaled, feature_cols) for the isolation forest.
    Uses numeric + engineered one-hot columns only.
    """
    base_cols = [
        "event_id", "severity_value", "is_system",
        "process_depth", "hour_of_day", "day_of_week",
        "granted_access", "message_len",
    ]
    eid_cols = [c for c in df.columns if c.startswith("eid_")]
    feature_cols = base_cols + eid_cols

    X = df[feature_cols].fillna(0).astype(float).values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, feature_cols, scaler


def run_isolation_forest(df: pd.DataFrame, X_scaled: np.ndarray,
                          contamination: float = CONTAMINATION) -> pd.DataFrame:
    print("Training Isolation Forest...")
    iso = IsolationForest(
        contamination=contamination,
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    df = df.copy()
    df["anomaly_label"] = iso.fit_predict(X_scaled)          # -1 = anomaly
    df["anomaly_score"]  = iso.score_samples(X_scaled)       # lower = more anomalous
    df["is_anomaly"]     = (df["anomaly_label"] == -1).astype(int)

    n = df["is_anomaly"].sum()
    print(f"Anomalies detected: {n:,}  ({n / len(df) * 100:.2f}%)")
    return df


def run_umap(X_scaled: np.ndarray, df: pd.DataFrame,
             sample_size: int = UMAP_SAMPLE) -> "go.Figure":
    """UMAP 2-D projection of a random sample, coloured by anomaly score."""
    n = min(sample_size, len(df))
    idx = np.random.default_rng(RANDOM_STATE).choice(len(df), n, replace=False)

    print(f"UMAP on {n:,} samples...")
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        metric="euclidean",
        random_state=RANDOM_STATE,
        low_memory=True,
    )
    emb = reducer.fit_transform(X_scaled[idx])

    plot_df = pd.DataFrame({
        "x": emb[:, 0],
        "y": emb[:, 1],
        "is_anomaly"   : df["is_anomaly"].iloc[idx].values,
        "anomaly_score": df["anomaly_score"].iloc[idx].values,
        "event_id"     : df["event_id"].iloc[idx].values,
        "hostname"     : df["hostname"].iloc[idx].values,
        "image_base"   : df["image_base"].iloc[idx].values,
    })

    fig = px.scatter(
        plot_df, x="x", y="y",
        color="anomaly_score",
        color_continuous_scale=["#e94560", "#f5a623", "#0f3460", "#16213e"],
        symbol="is_anomaly",
        hover_data=["event_id", "hostname", "image_base"],
        title="UMAP — Isolation Forest Anomaly Scores",
        template="plotly_dark",
        opacity=0.55,
        height=650,
        labels={"anomaly_score": "IF Score (lower = more anomalous)",
                "is_anomaly": "Anomaly"},
    )
    fig.update_traces(marker_size=3)
    fig.update_layout(
        font_family="Inter, sans-serif",
        title_font_size=18,
        coloraxis_colorbar_title="Score",
    )
    return fig


def anomaly_stage(normalized_path: str = NORMALIZED_PARQUET,
                  anomalies_path: str   = ANOMALIES_PARQUET,
                  contamination: float  = CONTAMINATION) -> pd.DataFrame:
    """
    End-to-end Stage 2 entry point.
    Returns DataFrame with anomaly columns added.
    """
    os.makedirs(os.path.dirname(anomalies_path), exist_ok=True)

    print("Loading normalized parquet...")
    df = pd.read_parquet(normalized_path)
    print(f"    Shape: {df.shape}")

    X_scaled, feature_cols, _ = build_feature_matrix(df)
    df = run_isolation_forest(df, X_scaled, contamination)

    fig = run_umap(X_scaled, df)
    fig.show()

    anomalies_df = df[df["is_anomaly"] == 1].copy()
    anomalies_df.to_parquet(anomalies_path, index=False)
    print(f"Anomalies saved -> {anomalies_path}")

    # Summary table
    summary = (
        anomalies_df.groupby("event_id")
        .agg(count=("event_id", "size"),
             avg_score=("anomaly_score", "mean"),
             hosts=("hostname", "nunique"))
        .sort_values("avg_score")
        .head(15)
    )
    print("\nTop anomalous EventIDs:\n", summary.to_string())

    return anomalies_df

In [ ]:
anomalies_df = anomaly_stage(
    normalized_path = NORMALIZED_PARQUET,
    anomalies_path  = ANOMALIES_PARQUET,
    contamination   = CONTAMINATION,
)


In [ ]:
import pandas as pd

anomalies_df = pd.read_parquet(ANOMALIES_PARQUET)

# Top anomalous event types
top = (
    anomalies_df.groupby("event_id")
    .agg(count=("event_id","size"),
         avg_if_score=("anomaly_score","mean"),
         unique_hosts=("hostname","nunique"))
    .sort_values("avg_if_score")
    .head(10)
)
display(top)


---
## Stage 3 — BERTopic (Semantic Topic Modelling)

Runs BERTopic on the anomalous event corpus:
1. **Clean** log text (strip GUIDs, hex, paths, timestamps)
2. **Embed** with `all-MiniLM-L6-v2` on GPU
3. **Cluster** with HDBSCAN
4. **Label** topics with c-TF-IDF keywords
5. **Visualise** — bar chart, inter-topic map, heatmap


In [ ]:
"""
Stage 3: BERTopic — Topic Modelling on Anomalous Events
Cleans log text, embeds with sentence-transformers, clusters with HDBSCAN,
and visualises topics interactively.
"""
import os
import re
import nltk
import pandas as pd
from tqdm.auto import tqdm

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# ── Config ────────────────────────────────────────────────────────────────────
ANOMALIES_PARQUET = "/content/data/anomalies.parquet"
TOPICS_PARQUET    = "/content/data/anomalies_with_topics.parquet"
TOPIC_MODEL_DIR   = "/content/data/bertopic_model"
EMBED_MODEL       = "all-MiniLM-L6-v2"
MIN_CLUSTER_SIZE  = 15
TOP_N_WORDS       = 10
# ──────────────────────────────────────────────────────────────────────────────


# ── Text cleaning ─────────────────────────────────────────────────────────────
_GUID_RE      = re.compile(r"\{[0-9a-fA-F\-]{8,}\}")
_HEX_RE       = re.compile(r"\b0x[0-9a-fA-F]+\b")
_TS_RE        = re.compile(r"\d{4}-\d{2}-\d{2}[T ]\d{2}:\d{2}:\d{2}(?:\.\d+)?")
_PATH_RE      = re.compile(r"[A-Za-z]:\\(?:[^\s\r\n|,\\]+\\)*([^\s\r\n|,\\]+)")
_NUM_RE       = re.compile(r"\b\d+\b")
_WS_RE        = re.compile(r"\s+")


def _ensure_stopwords():
    try:
        from nltk.corpus import stopwords
        return set(stopwords.words("english"))
    except LookupError:
        nltk.download("stopwords", quiet=True)
        from nltk.corpus import stopwords
        return set(stopwords.words("english"))


STOP_WORDS = _ensure_stopwords()

# Windows / Sysmon noise words that add no semantic value
DOMAIN_NOISE = {
    "rulename", "utctime", "processguid", "processid", "image",
    "targetprocessid", "targetprocessguid", "sourcename", "channel",
    "keywords", "opcodevalue", "severityvalue", "eventreceivedtime",
    "sourcemodulename", "sourcemoduletype", "version", "task",
    "threadid", "recordnumber", "executionprocessid", "providerguid",
    "timestamp", "version", "none", "null", "true", "false",
}


def clean_log_text(text: str) -> str:
    """
    Strip technical noise (GUIDs, hex, paths, timestamps, numbers)
    and return a bag-of-meaningful-words string.
    """
    # Replace paths with just the exe/filename token
    text = _PATH_RE.sub(lambda m: " " + m.group(1).lower() + " ", text)
    text = _GUID_RE.sub(" ", text)
    text = _HEX_RE.sub(" hexval ", text)
    text = _TS_RE.sub(" ", text)
    text = _NUM_RE.sub(" ", text)
    # Keep only alpha
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = _WS_RE.sub(" ", text).strip().lower()

    tokens = [
        w for w in text.split()
        if w not in STOP_WORDS
        and w not in DOMAIN_NOISE
        and len(w) > 2
    ]
    return " ".join(tokens) if tokens else "unknown_event"


# ── BERTopic pipeline ─────────────────────────────────────────────────────────

def build_topic_model() -> BERTopic:
    umap_model = UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42,
        low_memory=True,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    )
    vectorizer = CountVectorizer(
        stop_words="english",
        min_df=2,
        ngram_range=(1, 2),
        max_features=10_000,
    )
    embedding_model = SentenceTransformer(EMBED_MODEL)

    return BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer,
        top_n_words=TOP_N_WORDS,
        calculate_probabilities=True,
        verbose=True,
    )


def topic_stage(anomalies_path: str = ANOMALIES_PARQUET,
                topics_path: str    = TOPICS_PARQUET,
                model_dir: str      = TOPIC_MODEL_DIR) -> tuple[pd.DataFrame, BERTopic]:
    """
    End-to-end Stage 3 entry point.
    Returns (annotated_df, topic_model).
    """
    os.makedirs(os.path.dirname(topics_path), exist_ok=True)
    os.makedirs(model_dir, exist_ok=True)

    print("Loading anomalies...")
    df = pd.read_parquet(anomalies_path)
    print(f"    Shape: {df.shape}")

    # ── Prepare corpus ────────────────────────────────────────────────────────
    print("Cleaning log text...")

    def _col(name: str) -> pd.Series:
        """Safely retrieve a column; return empty strings if absent."""
        return df[name].fillna("") if name in df.columns else pd.Series("", index=df.index)

    corpus_raw = (
        _col("message") + " " +
        _col("image_base") + " " +
        _col("target_image_base") + " " +
        _col("target_object").apply(
            lambda x: x.split("\\")[-1].lower() if isinstance(x, str) and x else ""
        )
    )
    docs = corpus_raw.apply(clean_log_text).tolist()
    print(f"    Corpus size: {len(docs):,} documents")
    print(f"    Sample doc : {docs[0][:120]}")

    # ── Fit BERTopic ──────────────────────────────────────────────────────────
    print("\nFitting BERTopic...")
    topic_model = build_topic_model()
    topics, probs = topic_model.fit_transform(docs)

    df = df.copy()
    df["topic"]       = topics
    df["topic_prob"]  = [float(p.max()) if hasattr(p, "max") else float(p)
                         for p in probs]

    n_topics = len(set(topics)) - (1 if -1 in topics else 0)
    print(f"\nDiscovered {n_topics} topics (topic -1 = noise/outliers)")

    # ── Topic info ────────────────────────────────────────────────────────────
    topic_info = topic_model.get_topic_info()
    print("\nTop topics:\n", topic_info.head(12).to_string(index=False))

    # ── Visualisations ────────────────────────────────────────────────────────
    print("\nGenerating visualisations...")

    fig_bar = topic_model.visualize_barchart(
        top_n_topics=min(12, n_topics), n_words=8
    )
    fig_bar.update_layout(template="plotly_dark",
                          title="Security Event Topics — Top Keywords")
    fig_bar.show()

    if n_topics >= 2:
        fig_map = topic_model.visualize_topics()
        fig_map.update_layout(template="plotly_dark",
                              title="Inter-topic Distance Map")
        fig_map.show()

        fig_heat = topic_model.visualize_heatmap()
        fig_heat.update_layout(template="plotly_dark",
                               title="Topic Similarity Heatmap")
        fig_heat.show()

    # ── Save ──────────────────────────────────────────────────────────────────
    df.to_parquet(topics_path, index=False)
    topic_model.save(os.path.join(model_dir, "model.pkl"))
    print(f"\nSaved annotated parquet -> {topics_path}")
    print(f"Saved BERTopic model    -> {model_dir}")

    return df, topic_model


def get_topic_summary(topic_model: BERTopic, top_n: int = 15) -> dict:
    """
    Returns {topic_id: 'keyword1, keyword2, …'} for the RAG query builder.
    """
    summary = {}
    for tid in topic_model.get_topic_info()["Topic"].tolist():
        if tid == -1:
            continue
        words = topic_model.get_topic(tid)
        if words:
            summary[tid] = ", ".join([w for w, _ in words[:top_n]])
    return summary

In [ ]:
df_a, topic_model = topic_stage(
    anomalies_path = ANOMALIES_PARQUET,
    topics_path    = TOPICS_PARQUET,
    model_dir      = TOPIC_MODEL_DIR,
)


In [ ]:
import pandas as pd

topics_df = pd.read_parquet(TOPICS_PARQUET)
kw_map    = get_topic_summary(topic_model, top_n=8)

topics_df["topic_keywords"] = topics_df["topic"].map(
    lambda t: kw_map.get(t, "unknown")
)
topics_df.to_parquet(TOPICS_PARQUET, index=False)

print("Topic keyword map (first 8 topics):")
for tid, kw in list(kw_map.items())[:8]:
    count = (topics_df["topic"] == tid).sum()
    print(f"  Topic {tid:3d} ({count:5,} events): {kw}")


---
## Stage 4a — Build RAG Knowledge Base (ChromaDB)

Downloads and indexes **5 cybersecurity knowledge sources**:

| Collection | Source | Content |
|---|---|---|
| `mitre_attack` | MITRE ATT&CK v14 | ~700 techniques + descriptions |
| `mitre_d3fend` | MITRE D3FEND | Defensive countermeasures |
| `mitre_car` | MITRE CAR | Detection analytics & pseudocode |
| `cisa_kev` | CISA KEV | Known exploited CVEs + required actions |
| `sigma_rules` | SigmaHQ | 3000+ YAML detection rules |

> This cell takes **5–15 min** (cloning Sigma is the slow part). Run once; ChromaDB persists to disk.


In [ ]:
"""
Stage 4a: RAG Knowledge Base Builder
Downloads and indexes 5 cybersecurity knowledge sources into ChromaDB.
Sources: MITRE ATT&CK, MITRE D3FEND, MITRE CAR, CISA KEV, SigmaHQ Rules
"""
import os, re, json, subprocess
import requests
import chromadb
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
CHROMA_DIR   = "/content/data/chroma_db"
EMBED_MODEL  = "all-MiniLM-L6-v2"
BATCH_SIZE   = 128
KB_DIR       = "/content/data/kb_raw"
# ──────────────────────────────────────────────────────────────────────────────

_emb_model: SentenceTransformer = None

def _get_embedder():
    global _emb_model
    if _emb_model is None:
        print("Loading embedding model...")
        _emb_model = SentenceTransformer(EMBED_MODEL)
    return _emb_model


def _get_client():
    os.makedirs(CHROMA_DIR, exist_ok=True)
    return chromadb.PersistentClient(path=CHROMA_DIR)


def _upsert_collection(client: chromadb.Client,
                       name: str,
                       docs: list[str],
                       ids: list[str],
                       metas: list[dict]):
    """Create or replace a ChromaDB collection and batch-embed docs."""
    try:
        client.delete_collection(name)
    except Exception:
        pass
    col = client.create_collection(name)
    embedder = _get_embedder()

    for i in tqdm(range(0, len(docs), BATCH_SIZE), desc=f"  Indexing {name}"):
        bd = docs[i:i+BATCH_SIZE]
        bi = ids[i:i+BATCH_SIZE]
        bm = metas[i:i+BATCH_SIZE]
        emb = embedder.encode(bd, show_progress_bar=False).tolist()
        col.add(documents=bd, ids=bi, embeddings=emb, metadatas=bm)

    print(f"  {name}: {col.count():,} docs indexed")
    return col


# ── Source 1: MITRE ATT&CK ───────────────────────────────────────────────────

def load_mitre_attack(client):
    print("\nMITRE ATT&CK v14...")
    url = ("https://raw.githubusercontent.com/mitre/cti/master/"
           "enterprise-attack/enterprise-attack.json")
    data = requests.get(url, timeout=120).json()

    docs, ids, metas = [], [], []
    for obj in data["objects"]:
        if obj.get("type") != "attack-pattern":
            continue
        tech_id, tactic = "", ""
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-attack":
                tech_id = ref.get("external_id", "")
        for phase in obj.get("kill_chain_phases", []):
            tactic = phase.get("phase_name", "")
        name = obj.get("name", "")
        desc = obj.get("description", "")[:1800]
        text = f"ATT&CK {tech_id} — {name}\nTactic: {tactic}\n{desc}"
        uid  = f"attack_{tech_id}_{obj['id'][-6:]}"
        docs.append(text); ids.append(uid)
        metas.append({"source": "mitre_attack", "tech_id": tech_id,
                      "tactic": tactic, "name": name})

    _upsert_collection(client, "mitre_attack", docs, ids, metas)


# ── Source 2: MITRE D3FEND ───────────────────────────────────────────────────

def load_d3fend(client):
    print("\nMITRE D3FEND...")
    url = "https://d3fend.mitre.org/api/technique/all.json"
    try:
        data = requests.get(url, timeout=60).json()
        techniques = data.get("techniques") or data.get("data") or []
    except Exception as e:
        print(f"  D3FEND fetch failed: {e}")
        return

    docs, ids, metas = [], [], []
    for t in techniques:
        tid   = t.get("id") or t.get("d3f_id") or "unknown"
        label = t.get("label") or t.get("name") or ""
        desc  = t.get("definition") or t.get("description") or ""
        text  = f"D3FEND {tid} — {label}\n{desc}"[:2000]
        uid   = f"d3fend_{re.sub(r'[^a-zA-Z0-9_]', '_', tid)}"
        docs.append(text); ids.append(uid)
        metas.append({"source": "d3fend", "d3fend_id": tid, "name": label})

    if docs:
        _upsert_collection(client, "mitre_d3fend", docs, ids, metas)
    else:
        print("  D3FEND returned 0 usable techniques.")


# ── Source 3: MITRE CAR ──────────────────────────────────────────────────────

def load_car(client):
    print("\nMITRE CAR analytics...")
    car_dir = os.path.join(KB_DIR, "car")
    if not os.path.exists(car_dir):
        subprocess.run(
            ["git", "clone", "--depth=1",
             "https://github.com/mitre-attack/car.git", car_dir],
            check=True, capture_output=True,
        )

    import yaml
    docs, ids, metas = [], [], []
    analytics_dir = os.path.join(car_dir, "analytics")
    if not os.path.exists(analytics_dir):
        print("  CAR analytics directory not found.")
        return

    for fname in os.listdir(analytics_dir):
        if not (fname.endswith(".yaml") or fname.endswith(".yml")):
            continue
        try:
            with open(os.path.join(analytics_dir, fname), "r", encoding="utf-8") as f:
                obj = yaml.safe_load(f)
            title = obj.get("title", fname)
            desc  = obj.get("description", "")
            impl  = " ".join(
                str(i.get("code", ""))
                for i in (obj.get("implementations") or [])
                if isinstance(i, dict)
            )
            text = f"CAR — {title}\n{desc}\nDetection logic: {impl}"[:2000]
            uid  = f"car_{fname.replace('.yaml','').replace('.yml','')[:60]}"
            docs.append(text); ids.append(uid)
            metas.append({"source": "mitre_car", "title": title, "file": fname})
        except Exception:
            continue

    if docs:
        _upsert_collection(client, "mitre_car", docs, ids, metas)


# ── Source 4: CISA KEV ───────────────────────────────────────────────────────

def load_cisa_kev(client):
    print("\nCISA Known Exploited Vulnerabilities...")
    url = ("https://www.cisa.gov/sites/default/files/feeds/"
           "known_exploited_vulnerabilities.json")
    data = requests.get(url, timeout=60).json()

    docs, ids, metas = [], [], []
    for v in data.get("vulnerabilities", []):
        cve  = v.get("cveID", "unknown")
        text = (
            f"CVE: {cve} | Vendor: {v.get('vendorProject','')} | "
            f"Product: {v.get('product','')} | "
            f"Vulnerability: {v.get('vulnerabilityName','')} | "
            f"Required Action: {v.get('requiredAction','')} | "
            f"Due Date: {v.get('dueDate','')} | "
            f"Notes: {v.get('notes','')}"
        )[:2000]
        uid = f"kev_{cve}"
        docs.append(text); ids.append(uid)
        metas.append({"source": "cisa_kev", "cve_id": cve,
                      "product": v.get("product", "")})

    _upsert_collection(client, "cisa_kev", docs, ids, metas)


# ── Source 5: SigmaHQ Rules ──────────────────────────────────────────────────

def load_sigma(client):
    print("\nSigmaHQ detection rules (sparse clone)...")
    sigma_dir = os.path.join(KB_DIR, "sigma")
    if not os.path.exists(sigma_dir):
        subprocess.run(
            ["git", "clone", "--depth=1", "--filter=blob:none", "--sparse",
             "https://github.com/SigmaHQ/sigma.git", sigma_dir],
            check=True, capture_output=True,
        )
        subprocess.run(
            ["git", "sparse-checkout", "set", "rules/"],
            cwd=sigma_dir, check=True, capture_output=True,
        )

    import yaml
    docs, ids, metas, seen = [], [], [], set()
    rules_dir = os.path.join(sigma_dir, "rules")

    for root, _, files in os.walk(rules_dir):
        for fname in files:
            if not fname.endswith(".yml"):
                continue
            uid = f"sigma_{fname[:60]}"
            if uid in seen:
                uid += f"_{len(seen)}"
            seen.add(uid)
            try:
                with open(os.path.join(root, fname), "r",
                          encoding="utf-8", errors="replace") as f:
                    raw = f.read()
                obj   = yaml.safe_load(raw) or {}
                title = obj.get("title", fname)
                desc  = obj.get("description", "")
                tags  = ", ".join(obj.get("tags") or [])
                detect = str(obj.get("detection") or "")
                text = (
                    f"Sigma Rule: {title}\n"
                    f"Tags: {tags}\n"
                    f"Description: {desc}\n"
                    f"Detection: {detect}"
                )[:2000]
                docs.append(text); ids.append(uid)
                metas.append({"source": "sigma", "title": title,
                               "file": fname, "tags": tags})
            except Exception:
                continue

    if docs:
        _upsert_collection(client, "sigma_rules", docs, ids, metas)


# ── Source 6: MSRC CVRF ──────────────────────────────────────────────────────

def load_msrc_cvrf(client):
    print("\nMicrosoft Security Update Summaries (MSRC CVRF) (2023+)...")
    url = "https://api.msrc.microsoft.com/cvrf/v3.0/updates"
    try:
        data = requests.get(url, timeout=60).json()
        updates = data.get("value", [])
    except Exception as e:
        print(f"  MSRC fetch failed: {e}")
        return

    docs, ids, metas = [], [], []
    for u in updates:
        id_str = u.get("ID", "")
        if not id_str:
            continue
        
        # Limit to 2023 onwards
        year_str = id_str[:4]
        try:
            if int(year_str) < 2023:
                continue
        except ValueError:
            continue

        doc_title = u.get("DocumentTitle", "")
        init_date = u.get("InitialReleaseDate", "")
        cvrf_url = u.get("CvrfUrl", "")

        text = (
            f"MSRC Update: {id_str} | "
            f"Title: {doc_title} | "
            f"Release Date: {init_date} | "
            f"CVRF URL: {cvrf_url}"
        )[:2000]

        uid = f"msrc_{id_str}"
        docs.append(text)
        ids.append(uid)
        metas.append({"source": "msrc_cvrf", "update_id": id_str, "title": doc_title})

    if docs:
        _upsert_collection(client, "msrc_cvrf", docs, ids, metas)


# ── Entry point ───────────────────────────────────────────────────────────────

def build_knowledge_base():
    """Download and index all knowledge sources."""
    os.makedirs(KB_DIR, exist_ok=True)
    client = _get_client()

    load_mitre_attack(client)
    load_d3fend(client)
    load_car(client)
    load_cisa_kev(client)
    load_sigma(client)
    load_msrc_cvrf(client)

    print("\nKnowledge base complete.")
    print(f"    Collections: {[c.name for c in client.list_collections()]}")
    return client


def query_all_collections(client: chromadb.Client,
                          query: str,
                          top_k: int = 5) -> str:
    """
    Semantic search across all indexed collections.
    Returns a single concatenated context string.
    """
    embedder = _get_embedder()
    q_emb = embedder.encode([query]).tolist()
    parts  = []

    for col in client.list_collections():
        try:
            n = min(top_k, col.count())
            if n == 0:
                continue
            res = col.query(query_embeddings=q_emb, n_results=n)
            for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
                src = meta.get("source", col.name).upper()
                parts.append(f"[{src}]\n{doc[:600]}")
        except Exception as e:
            print(f"  {col.name}: {e}")

    return "\n\n---\n\n".join(parts)

In [ ]:
chroma_client = build_knowledge_base()


In [ ]:
# Verify collections
chroma_client = _get_client()
print("ChromaDB collections:")
for col in chroma_client.list_collections():
    print(f"  {col.name:20s} → {col.count():,} docs")

# Quick test query
test_ctx = query_all_collections(chroma_client, "lsass process access credential dump", top_k=2)
print("\nTest query (lsass credential dump):")
print(test_ctx[:800])


---



In [ ]:
# Install Ollama
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Start the server in the background
import subprocess
import time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)

# Pull the model (you will see the progress bar here!)
!ollama pull llama3


---
## Stage 4b — LLM Threat Analysis (DSPy + Ollama + RAG)

### What happens per anomalous event:
1. **Build context** string from event fields
2. **Retrieve** relevant docs from all ChromaDB collections (semantic search)
3. **DSPy `ChainOfThought`** reasons step-by-step over context + retrieved docs
4. **Ollama `llama3`** generates structured output:
   - `threat_analysis` — what the threat is
   - `mitre_technique` — ATT&CK ID + name
   - `remediation_steps` — numbered list
   - `severity_rating` — Critical / High / Medium / Low

> **DSPy `BootstrapFewShot`** auto-optimises prompt selection using 2 APT29 labelled examples.


In [ ]:
"""
Stage 4b: LLM Analysis — DSPy + Ollama
Defines the DSPy signature, ChainOfThought module, and BootstrapFewShot
optimizer. Runs threat analysis on the top anomalous events per BERTopic cluster.
"""
import os, json
import pandas as pd
import dspy
import chromadb
from IPython.display import display, Markdown

try:
    from pipeline.stage4a_rag_kb import query_all_collections, _get_client, _get_embedder
except ImportError:
    pass  # In notebook mode, these functions are already in the global namespace

# ── Config ────────────────────────────────────────────────────────────────────
TOPICS_PARQUET  = "/content/data/anomalies_with_topics.parquet"
RESULTS_JSON    = "/content/data/llm_results.json"
OLLAMA_MODEL    = "llama3"           # swap to mistral / phi3 if preferred
OLLAMA_BASE_URL = "http://localhost:11434"
TOP_N_TOPICS    = 12                 # topics to analyse
EVENTS_PER_TOPIC = 3                 # worst-scoring events per topic
RAG_TOP_K       = 5
# ──────────────────────────────────────────────────────────────────────────────


# ── Ollama bootstrap ──────────────────────────────────────────────────────────

def configure_dspy(model: str = OLLAMA_MODEL):
    """Wire DSPy to the local Ollama endpoint."""
    lm = dspy.LM(
        model=f"ollama_chat/{model}",
        api_base=OLLAMA_BASE_URL,
        api_key="ollama",
        temperature=0.15,
        max_tokens=1024,
    )
    dspy.configure(lm=lm)
    print(f"  DSPy configured -> ollama/{model}")
    return lm


# ── DSPy Signature ────────────────────────────────────────────────────────────

class ThreatAnalysis(dspy.Signature):
    """
    You are a senior threat-intelligence analyst.
    Given a Windows security anomaly, related topic keywords, and retrieved
    threat-intel context, produce a structured analysis.
    """
    anomaly_context: str  = dspy.InputField(
        desc="EventID, process image, hostname, account, granted access, "
             "and the raw Sysmon message snippet."
    )
    topic_keywords: str   = dspy.InputField(
        desc="BERTopic cluster keywords that describe the group this event belongs to."
    )
    retrieved_docs: str   = dspy.InputField(
        desc="Relevant passages from MITRE ATT&CK, D3FEND, CAR, CISA KEV, "
             "and Sigma rules retrieved via semantic search."
    )

    threat_analysis: str    = dspy.OutputField(
        desc="1–3 sentence analysis of the likely threat this anomaly represents."
    )
    mitre_technique: str    = dspy.OutputField(
        desc="Most applicable MITRE ATT&CK technique, e.g. 'T1055 - Process Injection'."
    )
    remediation_steps: str  = dspy.OutputField(
        desc="Numbered list of 3–5 concrete detection or remediation steps."
    )
    severity_rating: str    = dspy.OutputField(
        desc="One of: Critical / High / Medium / Low — with a one-sentence rationale."
    )


# ── DSPy Module ───────────────────────────────────────────────────────────────

class SecurityAnalyzer(dspy.Module):
    def __init__(self):
        super().__init__()
        self.analyze = dspy.ChainOfThought(ThreatAnalysis)

    def forward(self, anomaly_context: str,
                topic_keywords: str,
                retrieved_docs: str) -> dspy.Prediction:
        return self.analyze(
            anomaly_context=anomaly_context,
            topic_keywords=topic_keywords,
            retrieved_docs=retrieved_docs,
        )


# ── Few-shot examples for BootstrapFewShot ───────────────────────────────────

FEW_SHOT_EXAMPLES = [
    dspy.Example(
        anomaly_context=(
            "EventID: 10 | Process: C:\\Windows\\System32\\lsass.exe | "
            "Target: lsass.exe | GrantedAccess: 0x1FFFFF | "
            "Message: Process accessed lsass with full handle rights."
        ),
        topic_keywords="lsass, credential, access, memory, dump, mimikatz, process",
        retrieved_docs="[MITRE_ATTACK] T1003.001 - LSASS Memory: Adversaries may "
                       "attempt to access credential material stored in LSASS.",
        threat_analysis=(
            "This event strongly indicates credential dumping via direct LSASS "
            "memory access, consistent with tools like Mimikatz or ProcDump. "
            "Full handle rights (0x1FFFFF) are rarely required by legitimate processes."
        ),
        mitre_technique="T1003.001 - OS Credential Dumping: LSASS Memory",
        remediation_steps=(
            "1. Enable Credential Guard (Windows 10/11).\n"
            "2. Restrict LSASS access via Protected Process Light (PPL).\n"
            "3. Alert on GrantedAccess 0x1FFFFF targeting lsass.exe.\n"
            "4. Deploy Sigma rule 'win_lsass_access_non_system_account'.\n"
            "5. Review process lineage for the accessing process."
        ),
        severity_rating="Critical — Direct credential theft enables lateral movement.",
    ).with_inputs("anomaly_context", "topic_keywords", "retrieved_docs"),

    dspy.Example(
        anomaly_context=(
            "EventID: 13 | Process: reg.exe | "
            "TargetObject: HKLM\\SOFTWARE\\Microsoft\\Windows\\CurrentVersion\\Run\\backdoor | "
            "Message: Registry value set for persistence."
        ),
        topic_keywords="registry, persistence, run, key, startup, autorun",
        retrieved_docs="[MITRE_ATTACK] T1547.001 - Registry Run Keys: Adversaries "
                       "may achieve persistence by adding a program to a Run key.",
        threat_analysis=(
            "A new Run key was written by reg.exe, a classic persistence mechanism. "
            "The key name 'backdoor' is highly suspicious and warrants immediate review."
        ),
        mitre_technique="T1547.001 - Boot or Logon Autostart: Registry Run Keys",
        remediation_steps=(
            "1. Remove the malicious Run key immediately.\n"
            "2. Alert on unexpected writes to HKLM\\SOFTWARE\\...\\Run\\.\n"
            "3. Audit reg.exe invocations not launched by administrators.\n"
            "4. Use Sigma rule 'win_registry_run_key_modification'.\n"
            "5. Investigate the parent process that spawned reg.exe."
        ),
        severity_rating="High — Persistence mechanism allows re-infection after reboot.",
    ).with_inputs("anomaly_context", "topic_keywords", "retrieved_docs"),
]


def optimize_analyzer(analyzer: SecurityAnalyzer,
                      examples: list = FEW_SHOT_EXAMPLES) -> SecurityAnalyzer:
    """Run BootstrapFewShot to auto-select best prompts."""
    print("  Running DSPy BootstrapFewShot optimisation...")
    optimizer = dspy.BootstrapFewShot(max_bootstrapped_demos=2,
                                      max_labeled_demos=2)
    # Metric: response is non-empty (adjust with a real eval if labels exist)
    def metric(example, pred, trace=None):
        return (bool(pred.threat_analysis) and
                bool(pred.mitre_technique) and
                bool(pred.remediation_steps))

    optimized = optimizer.compile(analyzer, trainset=examples, metric=metric)
    print("  Optimisation complete.")
    return optimized


# ── Context builder ───────────────────────────────────────────────────────────

def build_anomaly_context(row: pd.Series) -> str:
    ga = row.get("granted_access", 0)
    ga_hex = hex(int(ga)) if ga else "N/A"
    return (
        f"EventID: {row['event_id']} | "
        f"Hostname: {row['hostname']} | "
        f"Process: {row['image']} | "
        f"TargetImage: {row.get('target_image', '')} | "
        f"TargetObject: {str(row.get('target_object', ''))[:80]} | "
        f"Account: {row['account_name']} ({row['domain']}) | "
        f"GrantedAccess: {ga_hex} | "
        f"IsolationForest score: {row.get('anomaly_score', 'N/A'):.4f} | "
        f"Message: {str(row.get('message', ''))[:400]}"
    )


# ── Main analysis loop ────────────────────────────────────────────────────────

def display_result(result: dict, idx: int):
    md = f"""
---
### Analysis #{idx+1} — Topic {result['topic_id']}
**Event:** `{result['event_id']}` on `{result['hostname']}`
**Topic keywords:** _{result['topic_keywords']}_

**Threat Analysis**
{result['threat_analysis']}

**MITRE ATT&CK Technique**
`{result['mitre_technique']}`

**Remediation Steps**
{result['remediation_steps']}

**Severity:** {result['severity_rating']}
---
"""
    display(Markdown(md))


def llm_analysis_stage(topics_path: str    = TOPICS_PARQUET,
                       results_path: str   = RESULTS_JSON,
                       skip_optimize: bool = False) -> list[dict]:
    """
    End-to-end Stage 4b entry point.
    Returns list of result dicts, also saved to JSON.
    """
    # ── Setup ─────────────────────────────────────────────────────────────────
    configure_dspy(OLLAMA_MODEL)

    client   = _get_client()
    analyzer = SecurityAnalyzer()

    if not skip_optimize:
        try:
            analyzer = optimize_analyzer(analyzer)
        except Exception as e:
            print(f"  Optimisation skipped: {e}")

    # ── Load data ─────────────────────────────────────────────────────────────
    print(f"\n  Loading {topics_path}...")
    df = pd.read_parquet(topics_path)
    valid_topics = sorted(
        [t for t in df["topic"].unique() if t != -1],
        key=lambda t: df[df["topic"] == t]["anomaly_score"].mean()
    )[:TOP_N_TOPICS]

    print(f"  Analysing {len(valid_topics)} topics × {EVENTS_PER_TOPIC} events each...\n")

    results = []

    for topic_id in valid_topics:
        t_df = df[df["topic"] == topic_id].nsmallest(EVENTS_PER_TOPIC, "anomaly_score")

        # Get topic keywords from BERTopic (stored in df if available, else from model)
        topic_kw = df[df["topic"] == topic_id]["topic_keywords"].iloc[0] \
                   if "topic_keywords" in df.columns else f"topic_{topic_id}"

        for _, row in t_df.iterrows():
            ctx   = build_anomaly_context(row)
            query = f"{ctx} {topic_kw}"
            docs  = query_all_collections(client, query, top_k=RAG_TOP_K)

            try:
                pred = analyzer(
                    anomaly_context=ctx,
                    topic_keywords=topic_kw,
                    retrieved_docs=docs,
                )
                rec = {
                    "topic_id"         : int(topic_id),
                    "topic_keywords"   : topic_kw,
                    "event_id"         : int(row["event_id"]),
                    "hostname"         : row["hostname"],
                    "anomaly_score"    : float(row.get("anomaly_score", 0)),
                    "threat_analysis"  : pred.threat_analysis,
                    "mitre_technique"  : pred.mitre_technique,
                    "remediation_steps": pred.remediation_steps,
                    "severity_rating"  : pred.severity_rating,
                }
                results.append(rec)
                display_result(rec, len(results) - 1)

            except Exception as e:
                print(f"  Topic {topic_id} event failed: {e}")

    # ── Save results ──────────────────────────────────────────────────────────
    os.makedirs(os.path.dirname(results_path), exist_ok=True)
    with open(results_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)
    print(f"\n  Results saved -> {results_path}")
    print(f"    Total analyses: {len(results)}")
    return results

In [ ]:
results = llm_analysis_stage(
    topics_path   = TOPICS_PARQUET,
    results_path  = RESULTS_JSON,
    skip_optimize = False,   # set True to skip BootstrapFewShot (faster)
)


---
## Results Dashboard

Summary visualisations of the LLM analysis output.


In [ ]:
import json, pandas as pd
import plotly.express as px
import plotly.graph_objects as go

with open(RESULTS_JSON) as f:
    results = json.load(f)

res_df = pd.DataFrame(results)
res_df = res_df.rename(columns={"mitre_technique": "Attack Technique", "remediation_steps": "Fixes"})
display(res_df[["topic_id","event_id","hostname","Attack Technique","Fixes","severity_rating"]].head(20))


In [ ]:
# Severity breakdown
sev_counts = res_df["severity_rating"].str.extract(r"(Critical|High|Medium|Low)")[0].value_counts()
fig = px.pie(
    values=sev_counts.values,
    names=sev_counts.index,
    title="Severity Distribution of Detected Anomalies",
    color=sev_counts.index,
    color_discrete_map={"Critical":"#e94560","High":"#f5a623","Medium":"#f0e130","Low":"#1db954"},
    template="plotly_dark",
    hole=0.4,
)
fig.update_layout(height=420)
fig.show()


In [ ]:
# MITRE technique frequency
tech_counts = res_df["Attack Technique"].value_counts().head(12)
fig = px.bar(
    x=tech_counts.values,
    y=tech_counts.index,
    orientation="h",
    title="Most Frequent MITRE ATT&CK Techniques",
    template="plotly_dark",
    color=tech_counts.values,
    color_continuous_scale="reds",
    height=500,
    labels={"x":"Count","y":"Technique"},
)
fig.update_layout(showlegend=False, yaxis=dict(autorange="reversed"))
fig.show()


In [ ]:
# Per-topic severity heatmap
import numpy as np

pivot = res_df.assign(
    sev_num=res_df["severity_rating"].str.extract(r"(Critical|High|Medium|Low)")[0].map(
        {"Critical":4,"High":3,"Medium":2,"Low":1}
    )
).groupby(["topic_id","event_id"])["sev_num"].mean().unstack(fill_value=0)

fig = go.Figure(go.Heatmap(
    z=pivot.values,
    x=[str(c) for c in pivot.columns],
    y=[f"Topic {r}" for r in pivot.index],
    colorscale="Reds",
    colorbar_title="Severity",
))
fig.update_layout(
    title="Severity Heatmap — Topic × EventID",
    template="plotly_dark",
    height=max(300, len(pivot)*40),
)
fig.show()


---
## Pipeline Complete

All outputs saved to `/content/data/`:
| File | Description |
|---|---|
| `normalized.parquet` | All ~1M events, flat schema |
| `anomalies.parquet` | Anomalous events with IF scores |
| `anomalies_with_topics.parquet` | Anomalies + BERTopic labels |
| `chroma_db/` | Persistent vector store (5 collections) |
| `bertopic_model/` | Saved BERTopic model |
| `llm_results.json` | Structured LLM threat analysis per anomaly |
